In [ ]:
'''
!fuser -v /dev/nvidia*
# Look for the PID 150912. If it's there, kill it:
!kill -9 150912

'''

In [ ]:
import torch, gc

# Force clear ALL GPU memory
torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

# Check what is using memory
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory reserved : {torch.cuda.memory_reserved()/1e9:.2f} GB")

# Kill any existing model in memory
import ctypes
ctypes.CDLL("libcuda.so").cuInit(0)

# Restart GPU context completely
torch.cuda.synchronize()
torch.cuda.empty_cache()
gc.collect()

print(f"\nAfter cleanup:")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory reserved : {torch.cuda.memory_reserved()/1e9:.2f} GB")

In [ ]:
F3_PATHS = ['/data/Dutch Government_F3_entire_8bit seismic.segy']

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
import torch, gc, os, sys, glob

RGTNET  = "/apps/rgtNet"
PROJECT = "/data/RGT_Project"
sys.path.insert(0, RGTNET)

from models import net3d
from data.augments import Reshape, ToTensor, CropVolume
import utils

# ── Load trained model ────────────────────────────────────────────────────────
SESSIONS_DIR = f"{PROJECT}/train_sessions"
checkpoints  = sorted(
    glob.glob(f"{SESSIONS_DIR}/**/checkpoint/*.pth",
              recursive=True),
    key=lambda p: int(os.path.basename(p).replace(".pth",""))
)
best_ckpt  = checkpoints[-5]
last_epoch = os.path.basename(best_ckpt).replace(".pth","")
print(f"Using checkpoint: epoch {last_epoch}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = net3d.model({
    'input_channels'  : 1,
    'encoder_channels': 512,
    'decoder_channels': 16
}).to(device)
model.load_state_dict(torch.load(best_ckpt, map_location=device))
model.eval()
print(f"Model loaded ✅")

In [ ]:
!pip install segyio

In [ ]:
import segyio
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
F3_PATH = next((p for p in F3_PATHS
                if os.path.exists(p)), None)
if F3_PATH is None:
    print("F3 not found — check path")
else:
    print(f"F3 found: {F3_PATH}")

In [ ]:
'''
with segyio.open(F3_PATH, "r", ignore_geometry=False) as segyfile:
    # Get data cube
    cube = segyio.tools.cube(segyfile)

    plt.imshow(cube[len(segyfile.ilines)//2, :, :].T, cmap='seismic')
    plt.show()
'''

In [ ]:
with segyio.open(F3_PATH, ignore_geometry=True) as f:
    times_sec  = f.samples / 1000.0
    data       = segyio.collect(f.trace[:])
    iline_nums = f.attributes(segyio.TraceField.INLINE_3D)[:]
    xline_nums = f.attributes(segyio.TraceField.CROSSLINE_3D)[:]
    n_samples  = f.bin[segyio.BinField.Samples]

ilines_u = np.unique(iline_nums)
xlines_u = np.unique(xline_nums)
il_map   = {v: i for i, v in enumerate(ilines_u)}
xl_map   = {v: i for i, v in enumerate(xlines_u)}

cube = np.zeros((len(ilines_u), len(xlines_u), n_samples),
                dtype=np.float32)
for t in range(len(data)):
    cube[il_map[iline_nums[t]],
         xl_map[xline_nums[t]], :] = data[t]

n_il, n_xl, n_t = cube.shape
print(f"F3 cube: {cube.shape}  (IL × XL × T)")
print(f"IL: {ilines_u[0]}→{ilines_u[-1]}")
print(f"XL: {xlines_u[0]}→{xlines_u[-1]}")
print(f"T : {times_sec[0]*1000:.0f}→{times_sec[-1]*1000:.0f} ms")

In [ ]:
mid_il_plot = cube.shape[0] // 2
plt.imshow(cube[mid_il_plot, :, :].T, cmap='gray')
plt.show()

In [ ]:
mid_xl_plot = cube.shape[1] // 2
plt.imshow(cube[:,mid_xl_plot, :].T, cmap='gray')
plt.show()

In [ ]:
il_c = cube.shape[0] // 2
xl_c = 475
t_c  = 380

subvol = cube[il_c-100 : il_c+100,
              xl_c-100 : xl_c+100,
              t_c-64  : t_c+64 ]

sub_ilines = ilines_u[il_c-100 : il_c+100]
sub_xlines = xlines_u[xl_c-100 : xl_c+100]
sub_times  = times_sec[t_c-64  : t_c+64]

print(f"F3 sub-volume : {subvol.shape}  (IL × XL × T)")
print(f"IL  : {sub_ilines[0]} → {sub_ilines[-1]}")
print(f"XL  : {sub_xlines[0]} → {sub_xlines[-1]}")
print(f"Time: {sub_times[0]*1000:.0f} → {sub_times[-1]*1000:.0f} ms")

import matplotlib.pyplot as plt
clip = np.percentile(np.abs(subvol), 98)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(subvol[64,:, :].T, aspect="auto",
               cmap="gray", vmin=-clip, vmax=clip)
axes[0].set_title(f"F3 Inline {sub_ilines[64]}",
                  fontweight="bold")
axes[0].set_xlabel("Inline"); axes[0].set_ylabel("Time (samples)")

axes[1].imshow(subvol[:,64, :].T, aspect="auto",
               cmap="gray", vmin=-clip, vmax=clip)
axes[1].set_title(f"F3 Crossline {sub_xlines[64]}",
                  fontweight="bold")
axes[1].set_xlabel("Crossline"); axes[1].set_ylabel("Time (samples)")

plt.suptitle("F3 Sub-Volume — Input to RGT Network",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
def mea_std_norm(x):
    return (x - x.mean()) / (x.std() + 1e-8)

# Prepare input tensor
# subvol shape: (200, 200, 128) = (IL, XL, T)
# model expects: (batch, channel, T, XL, IL) = (1,1,128,200,200)
seis_input = mea_std_norm(subvol)
seis_input = seis_input.transpose(2, 1, 0)    # → (T, XL, IL)
seis_tensor = torch.from_numpy(
    seis_input[np.newaxis, np.newaxis, ...]    # → (1,1,T,XL,IL)
).float().to(device)

print(f"Input tensor: {seis_tensor.shape}")

with torch.no_grad():
    rgt_tensor = model(seis_tensor)

rgt_pred = rgt_tensor.squeeze().cpu().numpy()  # (T, XL, IL)
# Transpose back to (IL, XL, T)
rgt_vol  = rgt_pred.transpose(2, 1, 0)

print(f"RGT output shape: {rgt_vol.shape}  (IL × XL × T)")
print(f"RGT range       : [{rgt_vol.min():.4f}, {rgt_vol.max():.4f}]")

# Normalize for visualization
rgt_norm = (rgt_vol - rgt_vol.min()) / \
           (rgt_vol.max() - rgt_vol.min() + 1e-8)

# Save
#np.save(f"{PROJECT}/outputs/f3_rgt_prediction.npy", rgt_vol)
print("Saved ✅")

In [ ]:
mid  = 64
#clip = np.percentile(np.abs(seis_np[:, mid, :]), 98)

def norm_display(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)



fig, axes = plt.subplots(1, 2, figsize=(20, 7))

axes[0].imshow(subvol[64,:, :].T, aspect="auto",
               cmap="gray", vmin=-clip, vmax=clip)
axes[0].set_title("(a) Seismic Input",
                  fontweight="bold", fontsize=13)
axes[0].set_xlabel("Inline")
axes[0].set_ylabel("Time (samples)")


im2 = axes[1].imshow(rgt_norm[64,:, :].T, aspect="auto",
                     cmap="jet", vmin=0, vmax=1)
axes[1].set_title(f"(c) Predicted RGT — Epoch {last_epoch}",
                  fontweight="bold", fontsize=13)
axes[1].set_xlabel("Inline")
plt.colorbar(im2, ax=axes[1], shrink=0.85)

plt.suptitle(
    "Seismic | Predicted RGT",
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
#plt.savefig(f"{PROJECT}/outputs/prediction_vs_truth.png",dpi=150, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
import draw

# Transpose to (T, XL, IL) for draw.py
seis_draw = subvol.transpose(2, 1, 0)       # (T, XL, IL)
rgt_draw  = rgt_norm.transpose(2, 1, 0)     # (T, XL, IL)

print("(a) F3 Seismic Cube")
draw.draw_slice(
    seis_draw,
    x_slice=0, y_slice=0, z_slice=127,
    cmap='gray', clab='Amplitude'
)

In [ ]:
print("(b) Predicted RGT on F3 Real Data")
draw.draw_slice(
    rgt_draw,
    x_slice=0, y_slice=0, z_slice=127,
    cmap='jet', clab='RGT'
)

In [ ]:
print("(c) F3 Seismic + Predicted Horizons")
draw.draw_slice_surf(
    seis_draw,
    x_slice=0, y_slice=0, z_slice=127,
    cmap='gray', clab='Amplitude',
    #isofs=[0.09,0.2, 0.35, 0.5, 0.65, 0.8],
    isofs=[0.38,0.5],
    volume2=rgt_draw
)

In [ ]:
P_IL, P_XL, P_T = 200, 200, 128
 
# Fixed inline & time centre
il_c = cube.shape[0] // 2
t_c  = 380
 
# Crossline centres — each shifted by +200
xl_centres = [275, 475, 675]   # ← add/remove as needed
 
# Which inline slice to display (relative to patch, 0 to P_IL-1)
display_il = 64   # middle of patch = 100
 
print(f"Patch size     : {P_IL} × {P_XL} × {P_T}")
print(f"Inline centre  : {il_c}  (range {il_c-P_IL//2}→{il_c+P_IL//2})")
print(f"Time centre    : {t_c}   (range {t_c-P_T//2}→{t_c+P_T//2})")
print(f"XL centres     : {xl_centres}")
print(f"Display IL idx : {display_il} within each patch")

In [ ]:
def mea_std_norm(x):
    return (x - x.mean()) / (x.std() + 1e-8)
 
def predict_patch(model, patch_3d, device):
    """
    Predict RGT for a single (IL, XL, T) patch.
    Returns RGT in (IL, XL, T) layout.
    """
    normed = mea_std_norm(patch_3d)
    tensor = torch.from_numpy(
        normed.transpose(2, 1, 0)[np.newaxis, np.newaxis, ...]
    ).float().to(device)
 
    with torch.no_grad():
        out = model(tensor)
 
    rgt = out.squeeze().cpu().numpy().transpose(2, 1, 0)
    return rgt
 
 
patches = []
for xl_c in xl_centres:
    il_s = il_c - P_IL // 2
    xl_s = xl_c - P_XL // 2
    t_s  = t_c  - P_T  // 2
 
    # Bounds check
    il_e = il_s + P_IL
    xl_e = xl_s + P_XL
    t_e  = t_s  + P_T
    assert il_e <= cube.shape[0], f"IL out of bounds: {il_e} > {cube.shape[0]}"
    assert xl_e <= cube.shape[1], f"XL out of bounds: {xl_e} > {cube.shape[1]}"
    assert t_e  <= cube.shape[2], f"T out of bounds: {t_e} > {cube.shape[2]}"
 
    subvol = cube[il_s:il_e, xl_s:xl_e, t_s:t_e].copy()
    rgt    = predict_patch(model, subvol, device)
    rgt_n  = (rgt - rgt.min()) / (rgt.max() - rgt.min() + 1e-8)
 
    patches.append({
        'seis'   : subvol,
        'rgt'    : rgt,
        'rgt_n'  : rgt_n,
        'il_rng' : (ilines_u[il_s], ilines_u[il_e - 1]),
        'xl_rng' : (xlines_u[xl_s], xlines_u[xl_e - 1]),
        't_rng'  : (times_sec[t_s] * 1000, times_sec[t_e - 1] * 1000),
        'xl_c'   : xl_c,
    })
    print(f"  Patch XL={xl_c}: shape={subvol.shape}  "
          f"XL {xlines_u[xl_s]}→{xlines_u[xl_e-1]}  "
          f"RGT [{rgt.min():.3f}, {rgt.max():.3f}]")
 
torch.cuda.empty_cache()
print(f"\n{len(patches)} patches predicted ✅")
 

In [ ]:
n = len(patches)
fig, axes = plt.subplots(2, n, figsize=(7 * n, 10))
 
# Handle single-patch case (axes shape)
if n == 1:
    axes = axes.reshape(2, 1)
 
for j, p in enumerate(patches):
    seis_2d = p['seis'][display_il, :, :].T   # (T, XL) for imshow
    rgt_2d  = p['rgt_n'][display_il, :, :].T
 
    clip = np.percentile(np.abs(p['seis']), 98)
 
    # ── Top row: Seismic ──
    ax = axes[0, j]
    ax.imshow(seis_2d, aspect='auto', cmap='gray',
              vmin=-clip, vmax=clip)
    ax.set_title(f"Seismic — XL centre {p['xl_c']}\n"
                 f"XL {p['xl_rng'][0]}→{p['xl_rng'][1]}",
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Crossline')
    if j == 0:
        ax.set_ylabel('Time (samples)')
 
    # ── Bottom row: Predicted RGT ──
    ax = axes[1, j]
    im = ax.imshow(rgt_2d, aspect='auto', cmap='jet', vmin=0, vmax=1)
    # Overlay horizon contours
    #ax.contour(rgt_2d, np.linspace(0, 1, 25), colors='k', linewidths=0.4, alpha=0.6)
    ax.set_title(f"Predicted RGT — XL centre {p['xl_c']}",
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Crossline')
    if j == 0:
        ax.set_ylabel('Time (samples)')
    #plt.colorbar(im, ax=ax, shrink=0.85, label='RGT')
 
plt.suptitle(
    f"F3 Multi-Patch: Inline {p['il_rng'][0]}→{p['il_rng'][1]}, "
    f"Time {p['t_rng'][0]:.0f}→{p['t_rng'][1]:.0f} ms, "
    f"Epoch {last_epoch}",
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f"{PROJECT}/outputs/f3_multi_patch_xl_slide.png",
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
il_s = il_c - P_IL // 2
t_s  = t_c  - P_T  // 2
 
xl_start_positions = []
for p in patches:
    xs = p['xl_c'] - P_XL // 2
    xl_start_positions.append(xs)
 
xl_min = min(xl_start_positions)
xl_max = max(xl_start_positions) + P_XL
total_xl = xl_max - xl_min
 
# Allocate full strips
seis_strip = np.zeros((P_IL, total_xl, P_T), dtype=np.float32)
rgt_strip  = np.zeros((P_IL, total_xl, P_T), dtype=np.float32)
 
# Place each patch at its true crossline position
for p, xs in zip(patches, xl_start_positions):
    local = xs - xl_min
    seis_strip[:, local : local + P_XL, :] = p['seis']
    rgt_strip[:, local : local + P_XL, :]  = p['rgt']
 
# Normalise RGT for display
rgt_norm = (rgt_strip - rgt_strip.min()) / \
           (rgt_strip.max() - rgt_strip.min() + 1e-8)
 
print(f"Stitched seismic : {seis_strip.shape}  (IL × XL × T)")
print(f"Stitched RGT     : {rgt_strip.shape}")
print(f"XL range         : {xl_min} → {xl_max}")

In [ ]:
seis_2d = seis_strip[display_il, :, :].T   # (T, XL)
rgt_2d  = rgt_norm[display_il, :, :].T
 
clip = np.percentile(np.abs(seis_2d[seis_2d != 0]), 98)
 
fig, axes = plt.subplots(2, 1, figsize=(20, 10))
 
# Seismic
ax = axes[0]
ax.imshow(seis_2d, aspect='auto', cmap='gray',
          vmin=-clip, vmax=clip)
ax.set_title('Seismic (Stitched along Crossline)', fontweight='bold', fontsize=13)
ax.set_ylabel('Time (samples)')
 
# RGT
ax = axes[1]
im = ax.imshow(rgt_2d, aspect='auto', cmap='jet', vmin=0, vmax=1)
#ax.contour(rgt_2d, np.linspace(0, 1, 30),colors='k', linewidths=0.3, alpha=0.5)
ax.set_title('Predicted RGT (Stitched along Crossline)', fontweight='bold', fontsize=13)
ax.set_xlabel('Crossline')
ax.set_ylabel('Time (samples)')
#plt.colorbar(im, ax=ax, shrink=0.7, label='RGT')
 
plt.suptitle(
    f'F3 Crossline Stitch — IL {il_c}, T {t_c}, '
    f'{len(patches)} patches, Epoch {last_epoch}',
    fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PROJECT}/outputs/f3_crossline_stitched.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Stitched figure saved ✅")

In [ ]:
display_xl = 64   
fig, axes = plt.subplots(2, n, figsize=(7 * n, 10))
if n == 1:
    axes = axes.reshape(2, 1)
 
for j, p in enumerate(patches):
    seis_2d = p['seis'][:, display_xl, :].T   # (T, IL)
    rgt_2d  = p['rgt_n'][:, display_xl, :].T
 
    clip = np.percentile(np.abs(p['seis']), 98)
 
    ax = axes[0, j]
    ax.imshow(seis_2d, aspect='auto', cmap='gray',
              vmin=-clip, vmax=clip)
    ax.set_title(f"Seismic — XL centre {p['xl_c']}\n"
                 f"(crossline section at mid-XL)",
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Inline')
    if j == 0:
        ax.set_ylabel('Time (samples)')
 
    ax = axes[1, j]
    im = ax.imshow(rgt_2d, aspect='auto', cmap='jet', vmin=0, vmax=1)
    #ax.contour(rgt_2d, np.linspace(0, 1, 25),colors='k', linewidths=0.4, alpha=0.6)
    ax.set_title(f"Predicted RGT — XL centre {p['xl_c']}",
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Inline')
    if j == 0:
        ax.set_ylabel('Time (samples)')
    #plt.colorbar(im, ax=ax, shrink=0.85, label='RGT')
 
plt.suptitle(
    f"F3 Multi-Patch (Crossline View): "
    f"Time {p['t_rng'][0]:.0f}→{p['t_rng'][1]:.0f} ms, "
    f"Epoch {last_epoch}",
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f"{PROJECT}/outputs/f3_multi_patch_xl_slide_xline_view.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Crossline view saved ✅")

In [ ]:
P_IL, P_XL, P_T = 200, 200, 128
 
# Fixed inline & crossline centre
il_c = cube.shape[0] // 2
xl_c = 475
 
# Time centres — sliding downward by +128 (or any step you want)
t_centres = [128, 256, 384]   # ← adjust as needed
 
# Which inline slice to display (relative to patch)
display_il = P_IL // 2
 
# Sanity check
for tc in t_centres:
    t_s, t_e = tc - P_T // 2, tc + P_T // 2
    assert t_s >= 0 and t_e <= cube.shape[2], \
        f"Time out of bounds: t_c={tc} → [{t_s}, {t_e}], max={cube.shape[2]}"
 
print(f"Patch size   : {P_IL} × {P_XL} × {P_T}")
print(f"IL centre    : {il_c}")
print(f"XL centre    : {xl_c}")
print(f"Time centres : {t_centres}")
print(f"F3 cube      : {cube.shape}  (max T index = {cube.shape[2]-1})")

In [ ]:
def mea_std_norm(x):
    return (x - x.mean()) / (x.std() + 1e-8)
 
def predict_patch(model, patch_3d, device):
    normed = mea_std_norm(patch_3d)
    tensor = torch.from_numpy(
        normed.transpose(2, 1, 0)[np.newaxis, np.newaxis, ...]
    ).float().to(device)
    with torch.no_grad():
        out = model(tensor)
    return out.squeeze().cpu().numpy().transpose(2, 1, 0)
 
 
patches = []
for tc in t_centres:
    il_s = il_c - P_IL // 2
    xl_s = xl_c - P_XL // 2
    t_s  = tc   - P_T  // 2
 
    subvol = cube[il_s : il_s + P_IL,
                  xl_s : xl_s + P_XL,
                  t_s  : t_s  + P_T].copy()
 
    rgt   = predict_patch(model, subvol, device)
    rgt_n = (rgt - rgt.min()) / (rgt.max() - rgt.min() + 1e-8)
 
    patches.append({
        'seis'  : subvol,
        'rgt_n' : rgt_n,
        't_c'   : tc,
        't_rng' : (times_sec[t_s] * 1000, times_sec[t_s + P_T - 1] * 1000),
        't_s'   : t_s,
    })
    print(f"  Patch t_c={tc}: T samples {t_s}→{t_s+P_T-1}  "
          f"({patches[-1]['t_rng'][0]:.0f}→{patches[-1]['t_rng'][1]:.0f} ms)  "
          f"RGT [{rgt.min():.3f}, {rgt.max():.3f}]")
 
torch.cuda.empty_cache()
print(f"\n{len(patches)} patches predicted ✅")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3 * n))
 
# Full vertical extent
t_min = min(p['t_s'] for p in patches)
t_max = max(p['t_s'] + P_T for p in patches)
full_t_len = t_max - t_min
 
# Seismic strip
seis_strip = np.full((full_t_len, P_XL), 0.0, dtype=np.float32)
rgt_strip  = np.full((full_t_len, P_XL), np.nan, dtype=np.float32)
 
for p in patches:
    local_start = p['t_s'] - t_min
    seis_2d = p['seis'][display_il, :, :].T   # (P_T, P_XL)
    rgt_2d  = p['rgt_n'][display_il, :, :].T
    seis_strip[local_start : local_start + P_T, :] = seis_2d
    rgt_strip[local_start : local_start + P_T, :]  = rgt_2d
 
clip = np.percentile(np.abs(seis_strip[seis_strip != 0]), 98)
 
ax = axes[0]
ax.imshow(seis_strip, aspect='auto', cmap='gray',
          vmin=-clip, vmax=clip,
          extent=[0, P_XL, t_max, t_min])
# Mark patch boundaries
for p in patches:
    ax.axhline(y=p['t_s'], color='lime', ls='--', lw=0.8, alpha=0.7)
    ax.axhline(y=p['t_s'] + P_T, color='lime', ls='--', lw=0.8, alpha=0.7)
ax.set_title('Seismic — Stitched Vertical', fontweight='bold')
ax.set_xlabel('Crossline')
ax.set_ylabel('Time (sample index)')
 
ax = axes[1]
im = ax.imshow(rgt_strip, aspect='auto', cmap='jet', vmin=0, vmax=1,
               extent=[0, P_XL, t_max, t_min])
for p in patches:
    ax.axhline(y=p['t_s'], color='k', ls='--', lw=0.8, alpha=0.5)
    ax.axhline(y=p['t_s'] + P_T, color='k', ls='--', lw=0.8, alpha=0.5)
ax.set_title('RGT — Stitched Vertical', fontweight='bold')
ax.set_xlabel('Crossline')
ax.set_ylabel('Time (sample index)')
plt.colorbar(im, ax=ax, shrink=0.8, label='RGT')
 
plt.suptitle(
    f"Vertical Stitch Preview — IL {il_c}, XL {xl_c}",
    fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PROJECT}/outputs/f3_vertical_stitch_preview.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Stitched view saved ✅")

In [ ]:
P_IL, P_XL, P_T = 200, 200, 128
 
# Fixed inline & crossline centre
il_c = cube.shape[0] // 2
xl_c = 475
 
# Overlap between adjacent time patches
T_OVERLAP = 32
 
# Time stride = P_T - T_OVERLAP = 96
T_STRIDE = P_T - T_OVERLAP
 
# Auto-generate time starts covering as much of the cube as possible
t_starts = []
t = 0
while t + P_T <= cube.shape[2]:
    t_starts.append(t)
    t += T_STRIDE
# Make sure last patch reaches the end
if t_starts[-1] + P_T < cube.shape[2]:
    t_starts.append(cube.shape[2] - P_T)
 
print(f"Patch size  : {P_IL} × {P_XL} × {P_T}")
print(f"Overlap     : {T_OVERLAP} samples")
print(f"Stride      : {T_STRIDE} samples")
print(f"Time starts : {t_starts}")
print(f"Num patches : {len(t_starts)}")
 
# Which inline slice to display
display_il = P_IL // 2

In [ ]:
def mea_std_norm(x):
    return (x - x.mean()) / (x.std() + 1e-8)
 
def predict_patch(model, patch_3d, device):
    normed = mea_std_norm(patch_3d)
    tensor = torch.from_numpy(
        normed.transpose(2, 1, 0)[np.newaxis, np.newaxis, ...]
    ).float().to(device)
    with torch.no_grad():
        out = model(tensor)
    return out.squeeze().cpu().numpy().transpose(2, 1, 0)
 
 
il_s = il_c - P_IL // 2
xl_s = xl_c - P_XL // 2
 
rgt_patches = []   # list of (IL, XL, T) arrays
seis_patches = []
 
for i, ts in enumerate(t_starts):
    subvol = cube[il_s : il_s + P_IL,
                  xl_s : xl_s + P_XL,
                  ts   : ts   + P_T].copy()
    rgt = predict_patch(model, subvol, device)
 
    seis_patches.append(subvol)
    rgt_patches.append(rgt)
 
    print(f"  Patch {i}: T=[{ts}, {ts+P_T})  "
          f"RGT [{rgt.min():.3f}, {rgt.max():.3f}]")
 
torch.cuda.empty_cache()
print(f"\n{len(rgt_patches)} patches predicted ✅")

In [ ]:
def merge_vertical(rgt_top, rgt_bot, ov):
    """
    Paper: "We estimate an RGT shift image by averaging the residual
    sum along the vertical axis in their overlapped regions. Then we
    use this shift image to correct the RGTs in every horizontal slice
    of both RGT sub-volumes."
    """
    ov_top = rgt_top[:, :, -ov:]
    ov_bot = rgt_bot[:, :, :ov]
 
    # shift image: mean residual over overlap
    shift = np.mean(ov_top - ov_bot, axis=2, keepdims=True)
 
    rgt_top_c = rgt_top - shift / 2.0
    rgt_bot_c = rgt_bot + shift / 2.0
 
    # smooth linear blend in overlap
    w = np.linspace(1, 0, ov, dtype=np.float32).reshape(1, 1, -1)
    blended = w * rgt_top_c[:, :, -ov:] + (1 - w) * rgt_bot_c[:, :, :ov]
 
    return np.concatenate([
        rgt_top_c[:, :, :-ov],
        blended,
        rgt_bot_c[:, :, ov:]
    ], axis=2)
 
 
# Merge all patches sequentially
print("Merging RGT patches ...")
merged_rgt = rgt_patches[0]
for i in range(1, len(rgt_patches)):
    ov = (t_starts[i-1] + P_T) - t_starts[i]
    ov = max(ov, 0)
    if ov > 0:
        merged_rgt = merge_vertical(merged_rgt, rgt_patches[i], ov)
    else:
        merged_rgt = np.concatenate([merged_rgt, rgt_patches[i]], axis=2)
    print(f"  After merging patch {i}: shape = {merged_rgt.shape}")
 
print(f"\nMerged RGT shape: {merged_rgt.shape}")
 
# Also stitch seismic the simple way (no merge needed, just place in position)
total_t = t_starts[-1] + P_T
merged_seis = np.zeros((P_IL, P_XL, total_t), dtype=np.float32)
for i, ts in enumerate(t_starts):
    merged_seis[:, :, ts:ts+P_T] = seis_patches[i]
 
# Trim both to same length
T_final = min(merged_rgt.shape[2], merged_seis.shape[2])
merged_rgt  = merged_rgt[:, :, :T_final]
merged_seis = merged_seis[:, :, :T_final]
 
print(f"Final seismic: {merged_seis.shape}")
print(f"Final RGT    : {merged_rgt.shape}")
 

In [ ]:
rgt_norm = (merged_rgt - merged_rgt.min()) / \
           (merged_rgt.max() - merged_rgt.min() + 1e-8)
 
seis_2d = merged_seis[display_il, :, :].T   # (T, XL)
rgt_2d  = rgt_norm[display_il, :, :].T
 
clip = np.percentile(np.abs(seis_2d), 98)
 
fig, axes = plt.subplots(1, 2, figsize=(16, 10))
 
# Seismic
ax = axes[0]
ax.imshow(seis_2d, aspect='auto', cmap='gray',
          vmin=-clip, vmax=clip)
ax.set_title('Seismic (Stitched)', fontweight='bold', fontsize=13)
ax.set_xlabel('Crossline')
ax.set_ylabel('Time (samples)')
 
# RGT — seamless
ax = axes[1]
im = ax.imshow(rgt_2d, aspect='auto', cmap='jet', vmin=0, vmax=1)
ax.contour(rgt_2d, np.linspace(0, 1, 30),colors='k', linewidths=0.3, alpha=0.5)
ax.set_title('Predicted RGT (Merged)', fontweight='bold', fontsize=13)
ax.set_xlabel('Crossline')
plt.colorbar(im, ax=ax, shrink=0.7, label='RGT')
 
plt.suptitle(
    f'F3 Vertical Merge — IL {il_c}, XL {xl_c}, '
    f'{len(t_starts)} patches, overlap={T_OVERLAP}',
    fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PROJECT}/outputs/f3_vertical_merged.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Merged figure saved ✅")
 

In [ ]:
P_IL, P_XL, P_T = 200, 200, 128
 
il_c = cube.shape[0] // 2
t_c  = 380
display_il = P_IL // 2
 
# ── Choose sliding direction ──────────────────────────────────────────────
# Set ONE of these to True
 
SLIDE_CROSSLINE = True
SLIDE_VERTICAL  = False
 
# Crossline config
XL_OVERLAP = 40
 
# Vertical config
T_OVERLAP = 32
 
print(f"Patch size : {P_IL} × {P_XL} × {P_T}")
print(f"Slide mode : {'Crossline' if SLIDE_CROSSLINE else 'Vertical'}")
 

In [ ]:
def compute_starts(full_len, patch_len, overlap):
    stride = patch_len - overlap
    starts = list(range(0, full_len - patch_len + 1, stride))
    if starts[-1] + patch_len < full_len:
        starts.append(full_len - patch_len)
    return starts
 
il_s = il_c - P_IL // 2
t_s  = t_c  - P_T  // 2
 
if SLIDE_CROSSLINE:
    slide_starts = compute_starts(cube.shape[1], P_XL, XL_OVERLAP)
    OVERLAP_SIZE = XL_OVERLAP
    print(f"XL starts  : {slide_starts}")
    print(f"Overlap    : {XL_OVERLAP}")
else:
    slide_starts = compute_starts(cube.shape[2], P_T, T_OVERLAP)
    OVERLAP_SIZE = T_OVERLAP
    print(f"T starts   : {slide_starts}")
    print(f"Overlap    : {T_OVERLAP}")
 
print(f"Num patches: {len(slide_starts)}")

In [ ]:
def mea_std_norm(x):
    return (x - x.mean()) / (x.std() + 1e-8)
 
def predict_patch(model, patch_3d, device):
    normed = mea_std_norm(patch_3d)
    tensor = torch.from_numpy(
        normed.transpose(2, 1, 0)[np.newaxis, np.newaxis, ...]
    ).float().to(device)
    with torch.no_grad():
        out = model(tensor)
    return out.squeeze().cpu().numpy().transpose(2, 1, 0)
 
rgt_list  = []
seis_list = []
 
for i, start in enumerate(slide_starts):
    if SLIDE_CROSSLINE:
        subvol = cube[il_s : il_s + P_IL,
                      start : start + P_XL,
                      t_s  : t_s  + P_T].copy()
    else:
        subvol = cube[il_s : il_s + P_IL,
                      (cube.shape[1]//2 - P_XL//2) : (cube.shape[1]//2 + P_XL//2),
                      start : start + P_T].copy()
 
    rgt = predict_patch(model, subvol, device)
    rgt_list.append(rgt)
    seis_list.append(subvol)
    print(f"  Patch {i}: start={start}  RGT [{rgt.min():.3f}, {rgt.max():.3f}]")
 
torch.cuda.empty_cache()
print(f"\n{len(rgt_list)} patches predicted ✅")

In [ ]:
def merge_vertical(rgt_top, rgt_bot, ov):
    """
    Vertical merge along axis 2 (time).
    Shift-correction: shift = mean(top_overlap - bot_overlap),
    then top -= shift/2, bot += shift/2, blend.
    """
    ov_top = rgt_top[:, :, -ov:]
    ov_bot = rgt_bot[:, :, :ov]
 
    shift = np.mean(ov_top - ov_bot, axis=2, keepdims=True)
 
    rgt_top_c = rgt_top - shift / 2.0
    rgt_bot_c = rgt_bot + shift / 2.0
 
    w = np.linspace(1, 0, ov, dtype=np.float32).reshape(1, 1, -1)
    blended = w * rgt_top_c[:, :, -ov:] + (1 - w) * rgt_bot_c[:, :, :ov]
 
    return np.concatenate([
        rgt_top_c[:, :, :-ov],
        blended,
        rgt_bot_c[:, :, ov:]
    ], axis=2)
 
 
def merge_horizontal(rgt_a, rgt_b, ov, axis=1):
    """
    Horizontal merge along given axis.
 
    CORRECTED: fits  f(rgt_b) ≈ rgt_a  in the overlap,
    i.e. learn how to map rgt_b values INTO rgt_a's scale.
    Then apply f to ALL of rgt_b.
 
    Uses simple linear fit:  rgt_a ≈ w0 + w1 * rgt_b
    (more stable than degree-2 polynomial)
    """
    # Move merge axis to front
    a = np.moveaxis(rgt_a, axis, 0)   # (merge_axis, ...)
    b = np.moveaxis(rgt_b, axis, 0)
 
    # Overlap values
    ov_a = a[-ov:].ravel().astype(np.float64)   # target
    ov_b = b[:ov].ravel().astype(np.float64)    # source to transform
 
    # Fit:  ov_a ≈ w0 + w1 * ov_b
    # Design matrix from rgt_b overlap (the source)
    X = np.column_stack([np.ones_like(ov_b), ov_b])
 
    # Least squares: (XᵀX)w = Xᵀ·ov_a
    XtX = X.T @ X
    Xty = X.T @ ov_a
    w = np.linalg.solve(XtX, Xty)
 
    print(f"    Merge axis={axis}: w0={w[0]:.4f}, w1={w[1]:.4f}  "
          f"(scale={w[1]:.3f}, shift={w[0]:.3f})")
 
    # Transform ALL of rgt_b
    fb = b.ravel().astype(np.float64)
    b_transformed = (w[0] + w[1] * fb).reshape(b.shape).astype(np.float32)
 
    # Linear blend in overlap
    blend_shape = [1] * a.ndim
    blend_shape[0] = ov
    blend_w = np.linspace(1, 0, ov, dtype=np.float32).reshape(blend_shape)
 
    blended = blend_w * a[-ov:] + (1 - blend_w) * b_transformed[:ov]
 
    merged = np.concatenate([a[:-ov], blended, b_transformed[ov:]], axis=0)
 
    # Move axis back
    return np.moveaxis(merged, 0, axis)
 

In [ ]:
print("Merging patches ...")
merged_rgt = rgt_list[0]
 
for i in range(1, len(rgt_list)):
    ov = (slide_starts[i-1] + (P_XL if SLIDE_CROSSLINE else P_T)) - slide_starts[i]
    ov = max(ov, 0)
 
    if ov > 0:
        if SLIDE_CROSSLINE:
            merged_rgt = merge_horizontal(merged_rgt, rgt_list[i], ov, axis=1)
        else:
            merged_rgt = merge_vertical(merged_rgt, rgt_list[i], ov)
    else:
        ax = 1 if SLIDE_CROSSLINE else 2
        merged_rgt = np.concatenate([merged_rgt, rgt_list[i]], axis=ax)
 
    print(f"  After patch {i}: shape = {merged_rgt.shape}")
 
# Stitch seismic (just place directly)
if SLIDE_CROSSLINE:
    total = slide_starts[-1] + P_XL
    merged_seis = np.zeros((P_IL, total, P_T), dtype=np.float32)
    for i, s in enumerate(slide_starts):
        merged_seis[:, s:s+P_XL, :] = seis_list[i]
    # Trim
    L = min(merged_rgt.shape[1], merged_seis.shape[1])
    merged_rgt = merged_rgt[:, :L, :]
    merged_seis = merged_seis[:, :L, :]
else:
    total = slide_starts[-1] + P_T
    xl_c_vert = cube.shape[1] // 2
    merged_seis = np.zeros((P_IL, P_XL, total), dtype=np.float32)
    for i, s in enumerate(slide_starts):
        merged_seis[:, :, s:s+P_T] = seis_list[i]
    L = min(merged_rgt.shape[2], merged_seis.shape[2])
    merged_rgt = merged_rgt[:, :, :L]
    merged_seis = merged_seis[:, :, :L]
 
# Global normalisation — key for seamless appearance
rgt_norm = (merged_rgt - merged_rgt.min()) / \
           (merged_rgt.max() - merged_rgt.min() + 1e-8)
 
print(f"\nMerged seismic: {merged_seis.shape}")
print(f"Merged RGT    : {merged_rgt.shape}")
print(f"RGT range     : [{merged_rgt.min():.4f}, {merged_rgt.max():.4f}]")

In [ ]:
seis_2d = merged_seis[display_il, :, :].T   # (T, XL or T_total)
rgt_2d  = rgt_norm[display_il, :, :].T
 
clip = np.percentile(np.abs(seis_2d[seis_2d != 0]), 98)
 
slide_label = 'Crossline' if SLIDE_CROSSLINE else 'Time'
 
fig, axes = plt.subplots(2, 1, figsize=(20, 10))
 
ax = axes[0]
ax.imshow(seis_2d, aspect='auto', cmap='gray', vmin=-clip, vmax=clip)
ax.set_title(f'Seismic (Stitched along {slide_label})',
             fontweight='bold', fontsize=13)
ax.set_ylabel('Time (samples)')
 
ax = axes[1]
im = ax.imshow(rgt_2d, aspect='auto', cmap='jet', vmin=0, vmax=1)
ax.contour(rgt_2d, np.linspace(0, 1, 30), colors='k', linewidths=0.3, alpha=0.5)
ax.set_title(f'Predicted RGT (Merged along {slide_label})',
             fontweight='bold', fontsize=13)
ax.set_xlabel(slide_label)
ax.set_ylabel('Time (samples)')
#plt.colorbar(im, ax=ax, shrink=0.7, label='RGT')
 
plt.suptitle(
    f'F3 {slide_label} Merge — {len(slide_starts)} patches, '
    f'overlap={OVERLAP_SIZE}, Epoch {last_epoch}',
    fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PROJECT}/outputs/f3_{slide_label.lower()}_merged_fixed.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Merged figure saved ✅")
 